In [77]:
import pandas as pd

matches = pd.read_csv("../data/gold_match.csv")
match_goals = pd.read_csv("../data/gold_match_goals.csv")
match_tickets = pd.read_csv("../data/gold_match_tickets.csv")
google_trends = pd.read_csv("../data/gold_google_trends_daily.csv")
match_context = pd.read_csv("../data/gold_match_context.csv")

In [78]:
# matches = matches.dropna("")
# pd.merge(matches, google_trends, on="match_id")
matches = matches.dropna(subset=["tickets_scanned"])
matches = matches[matches["is_home_match"] == True]
data = matches[["away_team_code", "last_result_vs_opponent", "kickoff_time_local", 'tickets_scanned', "match_id"]]

In [79]:
def score_to_int(s):
    if not isinstance(s, str):
        return s
    result, score = s.split()
    a, b = map(int, score.split("-"))
    diff = a - b
    return diff

data["last_result_vs_opponent"] = data["last_result_vs_opponent"].fillna(0)
data["last_result_vs_opponent"] =  data["last_result_vs_opponent"].apply(score_to_int)
data["weekday"] = match_context["weekday"]

In [80]:
data = data.reset_index(drop=True)

In [81]:
# data.corr()
# data.drop(["home_team_code", "away_team_code", "last_result_vs_opponent", "kickoff_time_local"], axis=1).corr()

pd.merge(data, google_trends, on="match_id")

,away_team_code,last_result_vs_opponent,kickoff_time_local,tickets_scanned,match_id,weekday,date,ohl_interest
0,WES,0,18:15:00,5565.0,d256yo3eng04m0fu7b4sl7wno,5,2022-07-30,17.32
1,CLU,-3,18:30:00,7440.0,d4mn5ksbxuvnaww4pmommxhqs,6,2022-08-14,30.38
2,KVO,2,18:15:00,4489.0,d65hmi7sq03yzr5he1k7ypus4,5,2022-08-27,22.18
3,CHA,3,20:45:00,4508.0,d80mkemezkz16bqh6lbn8tlhw,5,2022-09-10,19.08
4,STG,-3,16:00:00,6290.0,dak40etbhbqsr1nxyt50qcg0k,5,2022-10-01,17.75
...,...,...,...,...,...,...,...,...
66,CER,1,19:15:00,5812.0,ecz9iyxyc5ira9n59m8cp3bis,6,2025-12-21,34.16
67,STG,-5,20:45:00,5322.0,enw1n6mhkzvb6uptwhzs4gowk,5,2026-01-24,47.30
68,KVM,0,19:15:00,5971.0,eqncupz92qy89z9bxks1e0t90,6,2026-02-01,40.00
69,DEN,1,16:00:00,5137.0,ewgb5fczpfa71pfzihlip6yok,5,2026-02-14,40.00


In [82]:
# matches["stage"].unique()

# matches[matches["stage"] == "Conference League Play-off Group"]

data.drop(["away_team_code", "match_id", "kickoff_time_local"], axis=1).corr()

,last_result_vs_opponent,tickets_scanned,weekday
last_result_vs_opponent,1.000000,-0.096938,0.071193
tickets_scanned,-0.096938,1.000000,0.153500
weekday,0.071193,0.153500,1.000000


In [83]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded = encoder.fit_transform(data[['away_team_code']])

In [84]:
encoded = pd.DataFrame(encoded)

In [85]:
all_data = pd.concat([data, encoded], axis=1)

In [86]:
all_data.drop(["kickoff_time_local", "match_id", "away_team_code"], axis=1).corr()

,last_result_vs_opponent,tickets_scanned,weekday,0,1,2,3,4,5,6,...,11,12,13,14,15,16,17,18,19,20
last_result_vs_opponent,1.000000,-0.096938,0.071193,-0.066959,-0.057560,0.033690,0.000957,0.230591,-0.134874,0.137048,...,0.001189,0.166578,0.033690,0.033690,0.033690,0.254536,-0.406535,0.077582,0.057489,0.095311
tickets_scanned,-0.096938,1.000000,0.153500,0.365721,0.149926,-0.012457,-0.103161,-0.176970,0.277887,-0.191236,...,0.079595,-0.143074,-0.170391,0.055745,-0.095072,0.097888,0.009205,-0.067348,-0.058082,-0.062024
weekday,0.071193,0.153500,1.000000,0.093390,0.046948,-0.125028,0.035226,-0.025839,0.093390,-0.019718,...,0.019694,-0.011220,-0.011220,-0.011220,-0.125028,0.116125,0.035226,0.131409,-0.028522,0.065073
0,-0.066959,0.365721,0.093390,1.000000,-0.051321,-0.029204,-0.059701,-0.067252,-0.059701,-0.051321,...,-0.074235,-0.029204,-0.029204,-0.029204,-0.029204,-0.074235,-0.059701,-0.067252,-0.074235,-0.041599
1,-0.057560,0.149926,0.046948,-0.051321,1.000000,-0.025105,-0.051321,-0.057812,-0.051321,-0.044118,...,-0.063815,-0.025105,-0.025105,-0.025105,-0.025105,-0.063815,-0.051321,-0.057812,-0.063815,-0.035760
2,0.033690,-0.012457,-0.125028,-0.029204,-0.025105,1.000000,-0.029204,-0.032898,-0.029204,-0.025105,...,-0.036314,-0.014286,-0.014286,-0.014286,-0.014286,-0.036314,-0.029204,-0.032898,-0.036314,-0.020349
3,0.000957,-0.103161,0.035226,-0.059701,-0.051321,-0.029204,1.000000,-0.067252,-0.059701,-0.051321,...,-0.074235,-0.029204,-0.029204,-0.029204,-0.029204,-0.074235,-0.059701,-0.067252,-0.074235,-0.041599
4,0.230591,-0.176970,-0.025839,-0.067252,-0.057812,-0.032898,-0.067252,1.000000,-0.067252,-0.057812,...,-0.083624,-0.032898,-0.032898,-0.032898,-0.032898,-0.083624,-0.067252,-0.075758,-0.083624,-0.046860
5,-0.134874,0.277887,0.093390,-0.059701,-0.051321,-0.029204,-0.059701,-0.067252,1.000000,-0.051321,...,-0.074235,-0.029204,-0.029204,-0.029204,-0.029204,-0.074235,-0.059701,-0.067252,-0.074235,-0.041599
6,0.137048,-0.191236,-0.019718,-0.051321,-0.044118,-0.025105,-0.051321,-0.057812,-0.051321,1.000000,...,-0.063815,-0.025105,-0.025105,-0.025105,-0.025105,-0.063815,-0.051321,-0.057812,-0.063815,-0.035760


In [87]:
data["away_team_code"].unique()

<StringArray>
['WES', 'CLU', 'KVO', 'CHA', 'STG', 'GNK', 'GNT', 'SER', 'KVK', 'EUP', 'STV',
 'CER', 'ANT', 'ZWA', 'AND', 'KVM', 'STA', 'RWD', 'BEE', 'DEN', 'LAL']
Length: 21, dtype: str